In [ ]:
import os
import locale
import modal
from agents.preprocessor import Preprocessor
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
print(locale.getpreferredencoding())

In [ ]:
os.environ["PYTHONIOENCODING"] = "utf-8"

In [ ]:
preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(text)

In [ ]:
!uv run modal deploy -m pricer_service

In [ ]:
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()
reply = pricer.price.remote(text)
print(reply)

In [ ]:
reply = pricer.price.remote(text)
print(reply)

In [ ]:
from agents.specialist_agent import SpecialistAgent

In [ ]:
agent = SpecialistAgent()

In [ ]:
agent.price("iPhone 16")

In [ ]:
from dotenv import load_dotenv
import os
from huggingface_hub import Collection, login
from openai import OpenAI
import chromadb
from tqdm import tqdm
from agents.items import Item

In [ ]:
# Load the environment variables
load_dotenv(override=True)

# Global variables
VECTOR_DB = "products_vectorstore"

In [ ]:
db_client = chromadb.PersistentClient(path=VECTOR_DB)

In [ ]:
collection_name = "products"
existing_collection_names = [collection.name for collection in db_client.list_collections()]

In [ ]:
existing_collection_names

In [ ]:
if collection_name in existing_collection_names:
    db_client.delete_collection(name=collection_name)

In [ ]:
from openai import OpenAI

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"


In [ ]:
openai = OpenAI()
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi there!"}
]
resposne = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
print(resposne.choices[0].message.content)

In [ ]:
openai = OpenAI()
documents = ["documents to be embedded", "Another doucment which also needs to be embedded"]
responses = openai.embeddings.create(input=documents, model=EMBEDDING_MODEL)
embeddings = [item.embedding for item in responses.data]

In [ ]:
embeddings

In [ ]:
from dotenv import load_dotenv
import os
from huggingface_hub import Collection, login
from openai import OpenAI
import chromadb
from tqdm import tqdm
from agents.items import Item

In [ ]:
# Load the environment variables
load_dotenv(override=True)

# Global variables
VECTOR_DB = "products_vectorstore"
LITE_MODE = True
EMBEDDING_MODEL = "text-embedding-3-small"

In [ ]:
def get_data_from_huggingface() -> list[Item]:
    """ Login into huggingface and get data which will be loaded in vector database """

    # Login to huggingface
    print("Logging into Hugging Face...")
    hf_token = os.getenv("HF_TOKEN")
    login(token=hf_token, add_to_git_credential=False)

    print("Logged in. Fetching dataset...")

    # Get data from huggingface
    username = "ed-donner"
    dataset_name = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"
    train, val, test = Item.from_hub(dataset_name=dataset_name)

    print(f"Downloaded {len(train)} data from Hugging face.")

    return train

In [ ]:
db_client = chromadb.PersistentClient(path=VECTOR_DB)

In [ ]:
print(len(train))

In [ ]:
openai = OpenAI()

In [ ]:
# Check if the collection exists, if not create it
collection_name = "products"
existing_collection_names = [collection.name for collection in db_client.list_collections()]
existing_collection_names

In [ ]:
if collection_name in existing_collection_names:
    db_client.delete_collection(name=collection_name)

In [ ]:
db_client.delete_collection(name=collection_name)
collection = db_client.create_collection(name=collection_name)
for i in tqdm(range(0, len(train), 1000)):
    documents = [item.summary for item in train[i: i+1000]]
    response = openai.embeddings.create(input=documents, model=EMBEDDING_MODEL)
    embeddings = [item.embedding for item in response.data]
    print(embeddings)
    metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
    ids = [f"doc_{j}" for j in range(i, i+1000)]
    ids = ids[:len(documents)]
    collection.add(ids=ids, embeddings=embeddings, metadatas=metadatas, documents=documents)

In [6]:
from agents.frontier_agent import FrontierAgent
import chromadb
from dotenv import load_dotenv
from openai import OpenAI

In [7]:
# Load the environment variables
load_dotenv(override=True)

# Global variables
VECTOR_DB = "products_vectorstore"
LITE_MODE = True
EMBEDDING_MODEL = "text-embedding-3-small"
client = OpenAI()

In [8]:
collection_name = "products"
db_client = chromadb.PersistentClient(path=VECTOR_DB)
collection = db_client.get_or_create_collection(collection_name)

In [9]:
description = "Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal"

In [10]:
frontier_agent = FrontierAgent(collection=collection)
result = frontier_agent.price(description=description)
print(result)

229.0


In [11]:
from agents.specialist_agent import SpecialistAgent

In [12]:
specialist_agent = SpecialistAgent()
result = specialist_agent.price(description=description)
print(result)

209.0


In [13]:
from agents.neural_network_agent import NeuralNetworkAgent

In [14]:
network_agent_agent = NeuralNetworkAgent()
result = network_agent_agent.price(description=description)
print(result)

81.61827087402344


In [15]:
from agents.ensemble_agent import EnsembleAgent

In [16]:
ensemble_agent = EnsembleAgent(collection=collection)
result = ensemble_agent.price(description=description)
print(result)

97.25886248779298


In [1]:
from agents.scanner_agent import ScannerAgent

In [2]:
scanner_agent = ScannerAgent()
result = scanner_agent.scan()
print(result)

deals=[Deal(product_description='A MagSafe Qi2-compatible 15W car charger designed for iPhone 12–17 models, combining magnetic alignment with active cooling via a built-in fan to support sustained fast charging. The unit features strong suction mounting (rated up to 78+ lbs equivalent), dynamic RGB lighting with six modes, and Qi2/MagSafe compatibility for secure wireless attachment and efficient power delivery in automotive environments.', price=14.0, url='https://www.dealnews.com/Mag-Safe-Qi2-15-W-Car-Charger-With-Cooling-Fan-for-14-free-shipping-w-Prime/21816012.html?iref=rss-c142'), Deal(product_description='A compact 100W Anker Prime USB-C wall charger with two USB-C ports and one USB-A port, foldable prongs for portability, and high-power delivery suitable for charging laptops, tablets, and phones. Model A2688 supports multi-port charging while maintaining high output per port for devices that require up to 100W of power.', price=40.0, url='https://www.dealnews.com/products/Anker